# Exploración: yellow_tripdata_2026-03

**Propósito:** Análisis exploratorio del archivo raw para documentar
decisiones de transformación.

**Fecha:** 2026-10-02
**Autor:** Axl

## Objetivos

1. Entender la estructura y calidad del archivo raw
2. Identificar nulos sistemáticos y su causa
3. Detectar valores fuera de rango o sospechosos
4. Documentar las decisiones de transformación con evidencia

In [19]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = str(project_root / 'src')

In [20]:
sys.path = [p for p in sys.path if 'MLProjectTaxi' not in p]
sys.path.insert(0, src_path)

In [21]:
import pandas as pd
from pipelines.config import Config, TAXI_SCHEMAS
from pipelines.transform import transform_yellow

print("✅ Setup completo")
print(f"   project_root: {project_root}")
print(f"   sys.path[0]: {sys.path[0]}")

✅ Setup completo
   project_root: /media/axl/ExtraSpace1/Documents/python/physics/ComputationalPhysicsBook/MLProjectTaxi
   sys.path[0]: /media/axl/ExtraSpace1/Documents/python/physics/ComputationalPhysicsBook/MLProjectTaxi/src


In [22]:
RAW_PATH = Path("../data/raw/yellow_tripdata_2026-03.parquet")

df = pd.read_parquet(RAW_PATH)
print(f"Shape: {df.shape}")
print(f"Columnas: {len(df.columns)}")

Shape: (3952451, 20)
Columnas: 20


## 1. Vista general

Primer vistazo a la estructura del archivo: tipos de columnas, primeras 
filas, y estadísticas básicas.

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3952451 entries, 0 to 3952450
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

In [24]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.50,1.0,N,238,166,1,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.0,0.00
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2.0,0.88,1.0,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
3,2,2026-03-01 00:16:11,2026-03-01 00:28:20,1.0,1.76,1.0,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
4,2,2026-03-01 00:20:47,2026-03-01 00:30:44,2.0,1.57,1.0,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75


In [25]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,3952451.0,NaN,NaN,NaN,1.877893,1.0,2.0,2.0,2.0,7.0,0.724857
tpep_pickup_datetime,3952451,NaN,NaN,NaN,2026-03-16 09:09:15.947559,2008-12-31 23:03:20,2026-03-08 17:38:26,2026-03-16 08:13:05,2026-03-23 17:55:51.500000,2026-04-01 00:06:25,NaN
tpep_dropoff_datetime,3952451,NaN,NaN,NaN,2026-03-16 09:26:33.369682,2009-01-01 00:07:36,2026-03-08 17:55:32,2026-03-16 08:32:14,2026-03-23 18:13:19,2026-04-02 16:03:47,NaN
passenger_count,3006703.0,NaN,NaN,NaN,1.230823,0.0,1.0,1.0,1.0,8.0,0.633038
trip_distance,3952451.0,NaN,NaN,NaN,5.983693,0.0,1.01,1.82,3.71,288381.68,581.054487
RatecodeID,3006703.0,NaN,NaN,NaN,5.635411,1.0,1.0,1.0,1.0,99.0,20.591523
store_and_fwd_flag,3006703,2,N,3001767,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,3952451.0,NaN,NaN,NaN,161.508111,1.0,114.0,161.0,233.0,265.0,66.566407
DOLocationID,3952451.0,NaN,NaN,NaN,160.860248,1.0,107.0,162.0,234.0,265.0,70.707844
payment_type,3952451.0,NaN,NaN,NaN,0.880567,0.0,1.0,1.0,1.0,4.0,0.631797


## 2. Análisis de nulos

¿Cuántos nulos hay? ¿En qué columnas? ¿Son aleatorios o sistemáticos?

In [26]:
nulos = df.isna().sum().sort_values(ascending=False)
nulos = nulos[nulos > 0]
print(f"Total de nulos: {df.isna().sum().sum():,}")
print(f"Columnas con nulos: {len(nulos)}\n")
print(nulos)

Total de nulos: 4,728,740
Columnas con nulos: 5

passenger_count         945748
congestion_surcharge    945748
store_and_fwd_flag      945748
RatecodeID              945748
Airport_fee             945748
dtype: int64


In [27]:
pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
pct = pct[pct > 0]
print(pct.round(2))

passenger_count         23.93
congestion_surcharge    23.93
store_and_fwd_flag      23.93
RatecodeID              23.93
Airport_fee             23.93
dtype: float64


## 3. Hallazgo: nulos sistemáticos en 5 columnas

Cinco columnas tienen **exactamente** el mismo número de nulos: 945,748. 
Esto no es casualidad — sugiere que es el **mismo conjunto de filas** 
las que tienen nulos en todas ellas.

**Hipótesis:** estas filas vienen de un tipo de servicio que no reporta 
esos campos.

In [28]:
cols_con_nulos = [
    'passenger_count',
    'congestion_surcharge',
    'store_and_fwd_flag',
    'RatecodeID',
    'Airport_fee',
]

# ¿Todas tienen el mismo conteo?
for col in cols_con_nulos:
    print(f"{col}: {df[col].isna().sum():,}")

print(f"\nTotal filas: {len(df):,}")
print(f"Porcentaje: {df['passenger_count'].isna().sum() / len(df) * 100:.2f}%")

passenger_count: 945,748
congestion_surcharge: 945,748
store_and_fwd_flag: 945,748
RatecodeID: 945,748
Airport_fee: 945,748

Total filas: 3,952,451
Porcentaje: 23.93%


In [29]:
# ¿Las filas con nulos en passenger_count son las mismas que en las otras?
mask = df['passenger_count'].isna()

for col in cols_con_nulos:
    coinciden = (df[col].isna() == mask).all()
    print(f"{col}: {'✅ mismas filas' if coinciden else '❌ filas distintas'}")

passenger_count: ✅ mismas filas
congestion_surcharge: ✅ mismas filas
store_and_fwd_flag: ✅ mismas filas
RatecodeID: ✅ mismas filas
Airport_fee: ✅ mismas filas


## 4. Investigando la causa: payment_type

Sospechamos que esas filas comparten algún atributo. Revisamos las 
columnas categóricas clave.

In [30]:
sub = df[df['passenger_count'].isna()]

print("=== Filas con nulos ===")
print(sub[['VendorID', 'payment_type', 'RatecodeID']].value_counts(dropna=False))

=== Filas con nulos ===
VendorID  payment_type  RatecodeID
2         0             NaN           823584
1         0             NaN           112708
6         0             NaN             9456
Name: count, dtype: int64


In [31]:
print("=== Todo el dataset ===")
print(df['payment_type'].value_counts(dropna=False).sort_index())

=== Todo el dataset ===
payment_type
0     945748
1    2608830
2     353541
3      12842
4      31490
Name: count, dtype: int64


## 5. Conclusión: Flex Fare

**El 100% de las filas con nulos tienen `payment_type = 0`.**

Según el [NYC TLC Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf), 
`payment_type = 0` corresponde a **"Flex Fare trip"**, una modalidad de 
viaje E-Hail con tarifa fija no medida (upfront, non-metered).

Esto explica la ausencia de los campos anteriores:
- Al no pasar por taxímetro, no hay `RatecodeID` (tarifa medida)
- El conductor no ingresa `passenger_count`
- No hay `congestion_surcharge` ni `Airport_fee` porque la tarifa es fija
- `store_and_fwd_flag` no aplica porque no pasa por el sistema del vehículo

**Decisión:** excluir estas filas del dataset. La razón es de **alcance** 
(scope), no de calidad: el análisis se enfoca en viajes de yellow taxi 
con taxímetro. Flex Fare, aunque legítimo, tiene un esquema distinto 
que requeriría un pipeline separado.

**Impacto:** 945,748 filas excluidas (23.93% del total).

## 6. Valores fuera de rango

Después de excluir Flex Fare, revisamos si quedan valores sospechosos 
en el resto del dataset.

In [32]:
montos = ['fare_amount', 'tip_amount', 'total_amount', 'tolls_amount']

for col in montos:
    neg = (df[col] < 0).sum()
    if neg:
        print(f"{col}: {neg:,} negativos")

fare_amount: 20,724 negativos
tip_amount: 40 negativos
total_amount: 21,305 negativos
tolls_amount: 1,582 negativos


In [33]:
montos = ['fare_amount', 'tip_amount', 'total_amount', 'tolls_amount']

for col in montos:
    neg = (df[col] < 0).sum()
    if neg:
        print(f"{col}: {neg:,} negativos")

fare_amount: 20,724 negativos
tip_amount: 40 negativos
total_amount: 21,305 negativos
tolls_amount: 1,582 negativos


**Nota:** los montos negativos en NYC TLC son **legítimos**. Representan 
reembolsos, contracargos, o correcciones de viajes previos. No se dropean, 
pero se documentan.

In [34]:
print(f"trip_distance <= 0: {(df['trip_distance'] <= 0).sum():,}")
print(f"trip_distance == 0: {(df['trip_distance'] == 0).sum():,}")

trip_distance <= 0: 121,210
trip_distance == 0: 121,210


**Nota:** `trip_distance <= 0` puede indicar viajes cancelados (el taxi 
nunca se movió) o errores de captura. No se dropean por defecto, pero 
se documentan.

**Nota:** hay fechas fuera del rango del archivo (ej: 2008, 2026). 
Son errores de captura. Se documentan como deuda técnica para 
posible filtrado futuro.

In [35]:
invertidos = (df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']).sum()
print(f"Dropoff antes de pickup: {invertidos:,}")

Dropoff antes de pickup: 2


## 7. Decisiones documentadas

Resumen de las decisiones tomadas a partir de este análisis:

| Decisión | Razón | Impacto |
|----------|-------|---------|
| Excluir `payment_type == 0` | Flex Fare, fuera de alcance | -945,748 filas |
| Rellenar `passenger_count` con 0 | Default razonable | 0 nulos |
| Rellenar `ratecodeid` con 1 | Standard rate | 0 nulos |
| Rellenar montos con 0.0 | Sin monto registrado | 0 nulos |
| Rellenar `store_and_fwd_flag` con 'N' | Caso más común | 0 nulos |
| **NO** dropear montos negativos | Son legítimos (refunds) | - |
| **NO** dropear `trip_distance <= 0` | Pueden ser cancelaciones | - |

**Referencias:**
- [NYC TLC Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)
- `docs/decisions.md` — decisiones de diseño del pipeline

## 8. Resultado después de transform

Corremos el pipeline de transformación y verificamos el resultado final.

In [36]:
df_transformed = transform_yellow(df.copy(), TAXI_SCHEMAS['yellow'])

print(f"Shape final: {df_transformed.shape}")
print(f"Nulos totales: {df_transformed.isna().sum().sum()}")
print(f"\nTipos:")
print(df_transformed.dtypes)

2026-10-03 17:40:04,563 - pipelines.transform - INFO - Excluidas 945748 filas: Flex Fare, out of scope


Shape final: (3006703, 20)
Nulos totales: 0

Tipos:
vendorid                          int16
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                    int8
trip_distance                   float64
ratecodeid                         int8
store_and_fwd_flag             category
pulocationid                      int32
dolocationid                      int32
payment_type                      int16
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


## 9. Conclusiones

- El archivo contiene **3,952,451 filas** y 20 columnas
- **945,748 filas (23.93%)** son Flex Fare (`payment_type = 0`) y se excluyen
- Después del transform: **3,006,703 filas**, **0 nulos**
- Los tipos quedan consistentes con el schema
- Quedan valores "raros" (montos negativos, trip_distance <= 0) que son 
  legítimos y se documentan pero no se dropean

**Próximos pasos:**
- Tests unitarios para `transform_yellow`
- Generalizar para green, fhv, fhvhv
- Implementar Load